# 4. Figures & Tables

This notebook generates the four manuscript figures and the ablation analysis. All figures use a two-colour Okabe-Ito accessible palette (#D55E00 reject, #0072B2 approve) at 600 DPI.

In [ ]:
# Configuration
import json, sys, csv, hashlib, os
from pathlib import Path

# Deterministic matplotlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.dpi': 600,
    'savefig.dpi': 600,
    'font.family': 'sans-serif',
    'font.size': 10,
    'svg.hashsalt': 'eaa-p4',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})
np.random.seed(42)

def _repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config" / "harness_settings.json").is_file():
            return candidate
    return start

REPO_ROOT = _repo_root()
sys.path.insert(0, str(REPO_ROOT / 'engine'))
import corrected_public_engine_v1_1 as engine

CANONICAL_PATH = REPO_ROOT / 'data' / 'canonical' / 'canonical_dataset.json'
FIG_DIR = REPO_ROOT / 'outputs' / 'figures'
TBL_DIR = REPO_ROOT / 'outputs' / 'tables'

with open(CANONICAL_PATH) as f:
    canonical = json.load(f)

with open(REPO_ROOT / 'config' / 'presentation_config.json') as f:
    pres = json.load(f)

COL_REJECT = pres['figures']['palette']['reject']
COL_APPROVE = pres['figures']['palette']['approve']
COL_NEUTRAL = pres['figures']['palette']['neutral']
COL_HIGHLIGHT = pres['figures']['palette']['highlight']

orig12 = sorted(canonical['cases'].keys())
PROFILES = ['permissive', 'moderate', 'strict', 'very_strict']
gate_names = ['gate_safety', 'gate_evidence', 'gate_bias', 'gate_calibration', 'gate_traceability']
gate_labels = ['Safety', 'Evidence', 'Bias', 'Calibration', 'Traceability']

# Pre-compute Layer 1 moderate results
layer1 = {}
for cid in orig12:
    flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
    layer1[cid] = engine.evaluate_case({'case_id': cid, 'features': flat},
                                       profile_name='moderate', mode=engine.MODE_REPLAY)
print('Engine loaded, Layer 1 computed')

## 4.1 Figure 1 — Gate Failure Distribution & Governance Matrix

**Panel A**: Bar chart of gate failure counts. **Panel B**: Governance outcome matrix across 4 profiles × 12 cases.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.5]})

# Panel A: Gate failure counts (moderate, 11 rejected)
rejected = {cid: r for cid, r in layer1.items() if not r['approved']}
gate_counts = []
for g in gate_names:
    gate_counts.append(sum(1 for r in rejected.values() if not r.get(g, True)))

bars = ax1.barh(gate_labels, gate_counts, color=COL_REJECT, edgecolor='white')
ax1.set_xlabel('Number of cases failing gate (of 12)')
ax1.set_title('A. Gate Failure Distribution (Moderate Profile)')
ax1.set_xlim(0, 12)
for bar, count in zip(bars, gate_counts):
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             str(count), va='center', fontsize=9)

# Panel B: Governance outcome matrix (4 profiles x 12 cases)
matrix = []
for prof in PROFILES:
    row = []
    for cid in orig12:
        flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        r = engine.evaluate_case({'case_id': cid, 'features': flat},
                                 profile_name=prof, mode=engine.MODE_REPLAY)
        row.append(1 if r['approved'] else 0)
    matrix.append(row)

matrix_arr = np.array(matrix)
from matplotlib.colors import ListedColormap
cmap = ListedColormap([COL_REJECT, COL_APPROVE])
ax2.imshow(matrix_arr, cmap=cmap, aspect='auto', interpolation='nearest')

short_ids = [cid[:12] for cid in orig12]
ax2.set_xticks(range(12))
ax2.set_xticklabels(short_ids, rotation=45, ha='right', fontsize=7)
ax2.set_yticks(range(4))
ax2.set_yticklabels(['Permissive', 'Moderate', 'Strict', 'Very Strict'])
ax2.set_title('B. Governance Outcome Matrix')

# Add text annotations
for i in range(4):
    for j in range(12):
        txt = 'A' if matrix_arr[i, j] == 1 else 'R'
        ax2.text(j, i, txt, ha='center', va='center', fontsize=6,
                 color='white' if txt == 'R' else 'white')

plt.tight_layout()
fig.savefig(FIG_DIR / 'figure1_gate_failure.png', bbox_inches='tight',
            metadata={'Software': 'matplotlib', 'Creator': 'eaa-p4'})
plt.close()
print('F1: figure1_gate_failure.png ✓')

## 4.2 Figure 2 — Ablation Analysis

**Single gate removal**: no outcome changes (A1). **Pairwise removal**: three critical combinations (A2–A4).

In [ ]:
# Single gate ablation
single_changes = {}
for gi, gname in enumerate(gate_names):
    changes = 0
    for cid in orig12:
        flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        base_r = engine.evaluate_case({'case_id': cid, 'features': flat},
                                      profile_name='moderate', mode=engine.MODE_REPLAY)
        # Simulate removal by forcing gate to pass (set feature to safe value)
        ablated = dict(flat)
        # Map gate to its primary feature and set it to a passing value
        gate_feature_map = {
            'gate_safety': 'intrinsic_safety',
            'gate_evidence': 'evidence_strength',
            'gate_bias': 'bias_harm_index',
            'gate_calibration': 'uncertainty_calibration',
            'gate_traceability': 'traceability_integrity',
        }
        feat = gate_feature_map[gname]
        # Set to a value that guarantees pass (1.0 for higher-is-safer, 0.0 for higher-is-riskier)
        if feat == 'bias_harm_index':
            ablated[feat] = 0.0  # low = less biased = pass
        else:
            ablated[feat] = 1.0  # high = safer = pass
        abl_r = engine.evaluate_case({'case_id': cid, 'features': ablated},
                                     profile_name='moderate', mode=engine.MODE_REPLAY)
        if abl_r['approved'] != base_r['approved']:
            changes += 1
    single_changes[gname] = changes

print('Single gate removal:')
for g, c in single_changes.items():
    print(f'  Remove {g}: {c} outcome changes')
total_single = sum(single_changes.values())
assert total_single == 0, f'A1: Expected 0, got {total_single}'
print('A1: 0 single-gate changes ✓')

# Pairwise gate ablation
from itertools import combinations
pair_changes = {}
pair_flipped_cases = {}
for g1, g2 in combinations(range(5), 2):
    gn1, gn2 = gate_names[g1], gate_names[g2]
    key = f'{gate_labels[g1]}+{gate_labels[g2]}'
    changes = 0
    flipped = []
    for cid in orig12:
        flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        base_r = engine.evaluate_case({'case_id': cid, 'features': flat},
                                      profile_name='moderate', mode=engine.MODE_REPLAY)
        ablated = dict(flat)
        gate_feature_map = {
            'gate_safety': 'intrinsic_safety',
            'gate_evidence': 'evidence_strength',
            'gate_bias': 'bias_harm_index',
            'gate_calibration': 'uncertainty_calibration',
            'gate_traceability': 'traceability_integrity',
        }
        for gn in [gn1, gn2]:
            feat = gate_feature_map[gn]
            if feat == 'bias_harm_index':
                ablated[feat] = 0.0
            else:
                ablated[feat] = 1.0
        abl_r = engine.evaluate_case({'case_id': cid, 'features': ablated},
                                     profile_name='moderate', mode=engine.MODE_REPLAY)
        if abl_r['approved'] != base_r['approved']:
            changes += 1
            flipped.append(cid)
    pair_changes[key] = changes
    pair_flipped_cases[key] = flipped

print('\nPairwise gate removal:')
for k, c in sorted(pair_changes.items(), key=lambda x: -x[1]):
    if c > 0:
        print(f'  {k}: {c} changes ({pair_flipped_cases[k]})')

sb = pair_changes.get('Safety+Bias', 0)
sc = pair_changes.get('Safety+Calibration', 0)
et = pair_changes.get('Evidence+Traceability', 0)
assert sb == 3, f'A2: Expected 3, got {sb}'
assert sc == 2, f'A3: Expected 2, got {sc}'
assert et == 1, f'A4: Expected 1, got {et}'
print('\nA2: Safety+Bias=3 ✓, A3: Safety+Cal=2 ✓, A4: Evidence+Trace=1 ✓')

In [ ]:
# Draw Figure 2: Ablation heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 2]})

# Panel A: Single removal (all zeros)
single_vals = [single_changes[g] for g in gate_names]
ax1.barh(gate_labels, single_vals, color=COL_NEUTRAL, edgecolor='white')
ax1.set_xlabel('Outcome changes')
ax1.set_title('A. Single Gate Removal')
ax1.set_xlim(0, 4)
for i, v in enumerate(single_vals):
    ax1.text(v + 0.1, i, str(v), va='center', fontsize=9)

# Panel B: Pairwise heatmap
pair_matrix = np.zeros((5, 5))
for (g1, g2) in combinations(range(5), 2):
    key = f'{gate_labels[g1]}+{gate_labels[g2]}'
    pair_matrix[g1, g2] = pair_changes.get(key, 0)
    pair_matrix[g2, g1] = pair_changes.get(key, 0)

im = ax2.imshow(pair_matrix, cmap='YlOrRd', interpolation='nearest', vmin=0, vmax=4)
ax2.set_xticks(range(5))
ax2.set_xticklabels(gate_labels, rotation=45, ha='right')
ax2.set_yticks(range(5))
ax2.set_yticklabels(gate_labels)
ax2.set_title('B. Pairwise Gate Removal')
for i in range(5):
    for j in range(5):
        if i != j:
            ax2.text(j, i, int(pair_matrix[i, j]), ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax2, label='Outcome changes')

plt.tight_layout()
fig.savefig(FIG_DIR / 'figure2_ablation.png', bbox_inches='tight',
            metadata={'Software': 'matplotlib', 'Creator': 'eaa-p4'})
plt.close()
print('F2: figure2_ablation.png ✓')

## 4.3 Figure 3 — Provenance & Stability

**Panel A**: Provenance class distribution and confidence histogram. **Panel B**: Monte Carlo and perturbation stability summary.

In [ ]:
# Collect provenance and confidence data
prov_counts = {'direct_evidence': 0, 'rule_derived': 0, 'imputed_from_context': 0, 'uncertain_estimate': 0}
conf_vals = []
for cid in orig12:
    for fname, fobj in canonical['cases'][cid]['features'].items():
        pc = fobj.get('provenance_class', 'unknown')
        if pc in prov_counts:
            prov_counts[pc] += 1
        conf_vals.append(fobj.get('confidence_level', 0.0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Provenance + confidence
labels = ['Direct\nevidence', 'Rule\nderived', 'Imputed', 'Uncertain']
values = [prov_counts['direct_evidence'], prov_counts['rule_derived'],
          prov_counts['imputed_from_context'], prov_counts['uncertain_estimate']]
colors = [COL_APPROVE, COL_HIGHLIGHT, COL_NEUTRAL, COL_REJECT]
ax1.bar(labels, values, color=colors, edgecolor='white')
ax1.set_ylabel('Count (of 180 encodings)')
ax1.set_title('A. Provenance Distribution')
for i, v in enumerate(values):
    ax1.text(i, v + 1, f'{v} ({v/180*100:.0f}%)', ha='center', fontsize=8)

# Inset: confidence histogram
ax1_inset = ax1.inset_axes([0.55, 0.5, 0.4, 0.4])
ax1_inset.hist(conf_vals, bins=15, color=COL_APPROVE, alpha=0.7, edgecolor='white')
ax1_inset.axvline(0.591, color=COL_REJECT, linestyle='--', linewidth=1, label='Mean=0.591')
ax1_inset.set_xlabel('Confidence', fontsize=7)
ax1_inset.set_ylabel('Count', fontsize=7)
ax1_inset.tick_params(labelsize=6)
ax1_inset.legend(fontsize=6)

# Panel B: Stability summary
stability_metrics = ['Monte Carlo\n(12/12)', 'Perturbation\n(46/48)',
                     'Invariance\n(480/480)', 'Mode sens.\n(0 mod. div.)']
stability_pcts = [100.0, 95.8, 100.0, 100.0]
bar_colors = [COL_APPROVE if p == 100 else COL_HIGHLIGHT for p in stability_pcts]
ax2.bar(stability_metrics, stability_pcts, color=bar_colors, edgecolor='white')
ax2.set_ylabel('Stability (%)')
ax2.set_title('B. Decision Stability')
ax2.set_ylim(90, 101)
for i, v in enumerate(stability_pcts):
    ax2.text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
fig.savefig(FIG_DIR / 'figure3_provenance_stability.png', bbox_inches='tight',
            metadata={'Software': 'matplotlib', 'Creator': 'eaa-p4'})
plt.close()
print('F3: figure3_provenance_stability.png ✓')

## 4.4 Figure 4 — Compensation Effect

The non-compensatory framework rejects 11 of 12 cases. A hypothetical compensatory model would approve 2 additional cases — Google Flu Trends (0.57) and Uber AV (0.51).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

case_labels = [cid[:15] for cid in orig12]
nc_outcomes = [1 if layer1[cid]['approved'] else 0 for cid in orig12]
comp_scores = [layer1[cid].get('compensatory_score', 0) for cid in orig12]
comp_thresholds = [layer1[cid].get('compensatory_threshold', 0.5) for cid in orig12]
comp_outcomes = [1 if (cs >= ct if cs else False) else 0
                 for cs, ct in zip(comp_scores, comp_thresholds)]

x = np.arange(len(orig12))
width = 0.35

bars_nc = ax.bar(x - width/2, nc_outcomes, width, label='Non-compensatory',
                 color=COL_APPROVE, edgecolor='white')
bars_comp = ax.bar(x + width/2, comp_outcomes, width, label='Compensatory',
                   color=COL_HIGHLIGHT, edgecolor='white')

# Highlight divergence cases
for i, cid in enumerate(orig12):
    if nc_outcomes[i] != comp_outcomes[i]:
        ax.annotate(f'{comp_scores[i]:.2f}',
                    xy=(x[i] + width/2, comp_outcomes[i]),
                    xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=8, fontweight='bold', color=COL_REJECT)

ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Approved (1) / Rejected (0)')
ax.set_title('Non-Compensatory vs Compensatory Outcomes')
ax.legend()
ax.set_yticks([0, 1])
ax.set_yticklabels(['Rejected', 'Approved'])

plt.tight_layout()
fig.savefig(FIG_DIR / 'figure4_compensation.png', bbox_inches='tight',
            metadata={'Software': 'matplotlib', 'Creator': 'eaa-p4'})
plt.close()
print('F4: figure4_compensation.png ✓')

## 4.5 Write Ablation Matrix & Manifests

In [ ]:
# ablation_matrix.csv
abl_path = TBL_DIR / 'ablation_matrix.csv'
with open(abl_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['removal_type', 'gates_removed', 'outcome_changes', 'affected_cases'])
    for g in gate_names:
        w.writerow(['single', g, single_changes[g], ''])
    for key in sorted(pair_changes.keys()):
        if pair_changes[key] > 0:
            w.writerow(['pairwise', key, pair_changes[key],
                        ';'.join(pair_flipped_cases[key])])
print(f'Written: {abl_path}')

# Figure manifest
fig_files = sorted(FIG_DIR.glob('figure*.png'))
fm_path = FIG_DIR / 'paper4_figure_manifest.txt'
with open(fm_path, 'w') as f:
    f.write('Paper 4 Figure Manifest\n')
    for fp in fig_files:
        h = hashlib.sha256(fp.read_bytes()).hexdigest()
        f.write(f'{fp.name}  {h}\n')
print(f'Written: {fm_path}')

# Table manifest
tbl_files = sorted(TBL_DIR.glob('*.csv'))
tm_path = TBL_DIR / 'paper4_table_manifest.csv'
with open(tm_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['file', 'sha256'])
    for tp_f in tbl_files:
        if tp_f.name == 'paper4_table_manifest.csv':
            continue  # skip self
        h = hashlib.sha256(tp_f.read_bytes()).hexdigest()
        w.writerow([tp_f.name, h])
print(f'Written: {tm_path}')

print('\nAll figures and tables complete.')